# Решение: агент образовательных программ

Кейс GigaSchool «Точка сборки», задача 1. Этот ноутбук — то, что можно открыть и прогнать отдельно от экрана. Веб-студия в корне репозитория — то же решение для методиста: слева запрос, справа документ.

Порядок со слайдов кейса, его нельзя перепрыгивать:

1. Запрос свободным текстом и референсы: `.docx`, `.pdf`, текст.
2. Если не хватает аудитории, часов, шаблона или аттестации — вопросы, не больше пяти, и стоп.
3. Каркас: один из двух шаблонов АЦР, компетенции (знания, умения, навыки), учебный план. Часы тем, модулей и аттестации равны запрошенному объёму.
4. Методист правит или утверждает каркас. До этого тестов и лекций нет.
5. Комплект: вопросы с ключами, семинары, лекции. Каждый результат обучения закрыт вопросом.
6. Самопроверка. Источники только из приложенных файлов. Неподтверждённый источник помечается.

Модель — Cloud.ru Evolution. Зарубежные API не используются.


## Материалы кейса

Файлы лежат в `materials/`. Ниже читается текст docx без дополнительных библиотек.


In [1]:
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

ROOT = Path("..").resolve()
MATERIALS = ROOT / "materials"

def docx_text(path: Path) -> str:
    with zipfile.ZipFile(path) as archive:
        xml = archive.read("word/document.xml")
    root = ET.fromstring(xml)
    lines = []
    for paragraph in root.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}p"):
        text = "".join(node.text or "" for node in paragraph.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t"))
        if text.strip():
            lines.append(text.strip())
    return "\n".join(lines)

files = {
    "Общеразвивающая, 24 ч": MATERIALS / "program-general-24h.docx",
    "Повышение квалификации, 248 ч": MATERIALS / "program-qualification-248h.docx",
    "Пять открытых запросов": MATERIALS / "open-cases.docx",
}
for title, path in files.items():
    text = docx_text(path)
    print(f"{title}: {len(text)} знаков, файл на месте — {path.name}")


Общеразвивающая, 24 ч: 29256 знаков, файл на месте — program-general-24h.docx
Повышение квалификации, 248 ч: 27211 знаков, файл на месте — program-qualification-248h.docx
Пять открытых запросов: 9916 знаков, файл на месте — open-cases.docx


## Проверка часов

Сумма лекций, практик, лабораторных и самостоятельной работы равна строке модуля. Сумма модулей и итоговой аттестации равна объёму программы.

В образце на 248 часов шапка говорит «0 контактных, 246 самостоятельной, 2 аттестации», а итоговая строка плана — 66 практик, 6 лабораторных и 174 самостоятельной. Эту ошибку в новую программу не переносим.


In [2]:
def hours_ok(modules, attestation, duration):
    module_sum = 0
    notes = []
    for module in modules:
        kinds = module["lecture"] + module["practice"] + module["lab"] + module["self_study"]
        module_sum += module["hours"]
        if abs(kinds - module["hours"]) > 0.001:
            notes.append(f"{module['title']}: виды работы {kinds} ч, в строке {module['hours']} ч")
    total = module_sum + attestation
    if abs(total - duration) > 0.001:
        notes.append(f"итог {total} ч, запрошено {duration} ч")
    return not notes, notes

sample_24 = [
    {"title": "m1", "hours": 5, "lecture": 2, "practice": 2, "lab": 0, "self_study": 1},
    {"title": "m2", "hours": 6, "lecture": 2, "practice": 2, "lab": 0, "self_study": 2},
    {"title": "m3", "hours": 6, "lecture": 2, "practice": 3, "lab": 0, "self_study": 1},
    {"title": "m4", "hours": 7, "lecture": 2, "practice": 3, "lab": 0, "self_study": 2},
]
print("пример 24 ч", hours_ok(sample_24, 0, 24))

broken_248_header = [{"title": "стажировка", "hours": 246, "lecture": 0, "practice": 0, "lab": 0, "self_study": 246}]
broken_248_total = [{"title": "итоговая строка", "hours": 246, "lecture": 0, "practice": 66, "lab": 6, "self_study": 174}]
print("шапка 248 ч", hours_ok(broken_248_header, 2, 248))
print("ошибочная итоговая строка", hours_ok(broken_248_total, 2, 248))


пример 24 ч (True, [])
шапка 248 ч (True, [])
ошибочная итоговая строка (True, [])


## Пять открытых запросов

Скрытые запросы устроены так же. Ноутбук фиксирует, что именно должно сойтись.


In [3]:
CASES = [
    {"id": 1, "kind": "general", "hours": 16, "title": "Python для школьников", "must": "практики не меньше 8 ч, тест на 10 вопросов, слайды занятий 1 и 2"},
    {"id": 2, "kind": "qualification", "hours": 36, "title": "ИИ в работе учителя", "must": "удостоверение, профстандарт «Педагог», только сервисы, доступные в России"},
    {"id": 3, "kind": "qualification", "hours": 72, "title": "Анализ данных на Python", "must": "pandas, визуализация, SQL, статистика и вводный Python; слайды модуля про pandas"},
    {"id": 4, "kind": None, "hours": None, "title": "Кибербезопасность, запрос неполный", "must": "сначала не больше пяти вопросов, потом 72 ч повышения квалификации"},
    {"id": 5, "kind": "qualification", "hours": 256, "title": "Стажировка DevOps", "must": "254 ч самостоятельной работы, 2 ч аттестации, лекции и практики нули, ошибку 248 ч не копировать"},
]
for case in CASES:
    print(f"{case['id']}. {case['title']} — {case['must']}")


1. Python для школьников — практики не меньше 8 ч, тест на 10 вопросов, слайды занятий 1 и 2
2. ИИ в работе учителя — удостоверение, профстандарт «Педагог», только сервисы, доступные в России
3. Анализ данных на Python — pandas, визуализация, SQL, статистика и вводный Python; слайды модуля про pandas
4. Кибербезопасность, запрос неполный — сначала не больше пяти вопросов, потом 72 ч повышения квалификации
5. Стажировка DevOps — 254 ч самостоятельной работы, 2 ч аттестации, лекции и практики нули, ошибку 248 ч не копировать


## Вызов того же агента

Студия должна быть запущена: `npm run dev`. Ключ Cloud.ru лежит в `.env.local` и в git не попадает. Этот запрос только проверяет, что вход живой. Сборку программы делает чат на экране, после утверждения каркаса человеком.


In [11]:
import json
import urllib.request

payload = json.dumps({"messages": [], "course": {"brief": {"title": ""}}}).encode()
request = urllib.request.Request(
    "http://localhost:3000/api/chat",
    data=payload,
    headers={"Content-Type": "application/json"},
)
try:
    with urllib.request.urlopen(request, timeout=20) as response:
        print(response.status)
except Exception as error:
    print("Студия не ответила на localhost:3000.", error)
    print("Запустите npm run dev в корне репозитория. Если порт 3000 занят, сервер напишет другой адрес.")


200
